<div align="center">

<!-- MOTIONSALT branded banner. Rendered as HTML for the logo mark + gradient. -->
<div style="background:linear-gradient(135deg,#0f172a 0%,#1e1b4b 60%,#312e81 100%);padding:28px 24px;border-radius:14px;color:#f8fafc;font-family:-apple-system,Segoe UI,Roboto,sans-serif;">
  <div style="display:flex;align-items:center;justify-content:center;gap:14px;">
    <div style="width:44px;height:44px;border-radius:10px;background:linear-gradient(135deg,#22d3ee,#a855f7);display:flex;align-items:center;justify-content:center;font-weight:900;font-size:22px;color:#0f172a;">M</div>
    <div style="font-size:30px;font-weight:800;letter-spacing:2px;">MOTIONSALT</div>
  </div>
  <div style="margin-top:8px;font-size:14px;opacity:0.85;letter-spacing:3px;text-transform:uppercase;">Anime&nbsp;Video&nbsp;Upscaler</div>
  <div style="margin-top:14px;font-size:14px;opacity:0.75;max-width:640px;margin-left:auto;margin-right:auto;">A free, no-install, GPU-in-the-cloud alternative to Topaz Video AI. Powered by AnimeJaNai&nbsp;V3 and Real-ESRGAN AnimeVideo&nbsp;v3.</div>
  <div style="margin-top:18px;font-size:12px;opacity:0.7;">
    <a style="color:#a5f3fc;text-decoration:none;" href="https://github.com/motionssalt/upscale">github.com/motionssalt/upscale</a>
  </div>
</div>

</div>

---

**How this works:** step through the four cells below in order. Each cell is a self-contained step of a wizard — Connect ➜ Upload ➜ Configure ➜ Download. You never need to read or edit any code.

## Step 1 — Connect

Click **Connect** below. This verifies your GPU, installs the dependencies, and downloads the AI model weights from the MOTIONSALT GitHub Releases (never from HuggingFace — see the README for why).

In [ ]:
#@title 🔌 Step 1 — Connect { display-mode: "form" }
#@markdown Running this cell renders the **Connect** button immediately. Click it whenever you're ready.
import os, sys, subprocess, shutil, json, time, urllib.request, urllib.error
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

# ---------- MOTIONSALT global state ----------
MS = globals().setdefault("MOTIONSALT", {})
MS.setdefault("workdir", Path("/content/motionsalt"))
MS["workdir"].mkdir(parents=True, exist_ok=True)
MS.setdefault("weights_dir", MS["workdir"] / "weights")
MS["weights_dir"].mkdir(parents=True, exist_ok=True)
MS.setdefault("connected", False)

# ---------- Config: where to pull weights from ----------
GH_REPO   = "motionssalt/upscale"
GH_TAG    = "latest"
WEIGHTS = {
    "LOW":    "2x_AnimeJaNaiV3_SuperUltraCompact.pth",
    "MEDIUM": "2x_AnimeJaNaiV3_UltraCompact.pth",
    "HIGH":   "realesr-animevideov3.pth",
}
MS["weights_map"] = WEIGHTS
MS["gh_repo"]     = GH_REPO

# ---------- Plain-text emoji log (Fix #2: no HTML/CSS) ----------
# ipywidgets.Output is a real Jupyter output area — every print() lands here
# instantly and survives kernel restarts in the notebook JSON. This is the
# same idea as Step 3's logline() but promoted to the primary UI in every
# cell, so we don't depend on styled HTML rendering to see status.
log_out = widgets.Output(layout=widgets.Layout(
    border="1px solid #e2e8f0", padding="8px",
    max_height="360px", overflow="auto",
))
_ICON = {"ok":"✅","warn":"⚠️","err":"❌","run":"⏳","info":"•","perf":"📊","link":"🔗"}
def logline(kind, msg):
    icon = _ICON.get(kind, "•")
    line = f"{icon}  {msg}"
    with log_out:
        print(line, flush=True)
    # Mirror the important ones to stdout too, for post-hoc grep.
    if kind in ("ok","warn","err","perf","link"):
        print(f"[motionsalt/{kind}] {msg}", flush=True)

def section(title):
    with log_out:
        print("")
        print(f"── {title} ──", flush=True)

# ---------- The Connect button (live and clickable the moment this cell runs) ----------
btn = widgets.Button(
    description="Connect",
    icon="plug",
    button_style="primary",
    layout=widgets.Layout(width="180px", height="42px"),
)
status_lbl = widgets.Label(value="not connected")

def _run(cmd, quiet=True):
    p = subprocess.run(cmd, shell=isinstance(cmd,str), capture_output=True, text=True)
    if not quiet and p.returncode != 0:
        print(p.stdout[-2000:]); print(p.stderr[-2000:])
    return p.returncode, (p.stderr or p.stdout)[-400:]

def _resolve_release_tag():
    if GH_TAG != "latest":
        return GH_TAG
    url = f"https://api.github.com/repos/{GH_REPO}/releases/latest"
    with urllib.request.urlopen(url, timeout=20) as r:
        data = json.loads(r.read().decode("utf-8"))
    return data["tag_name"]

def _download(url, dest: Path, label: str):
    """Streaming download — progress reported as plain text heartbeats
    (no HTML progress bar; Fix #2)."""
    req = urllib.request.Request(url, headers={"User-Agent":"motionsalt-upscaler"})
    with urllib.request.urlopen(req, timeout=60) as r:
        total = int(r.headers.get("Content-Length", "0")) or 0
        read = 0
        last_pct = -1
        last_beat = time.time()
        with dest.open("wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk); read += len(chunk)
                now = time.time()
                if total:
                    pct = int(read * 100 / total)
                    if pct >= last_pct + 10 or (now - last_beat) >= 5.0:
                        logline("run", f"{label} … {pct}%  ({read/1e6:.1f}/{total/1e6:.1f} MB)")
                        last_pct = pct; last_beat = now
                else:
                    if (now - last_beat) >= 5.0:
                        logline("run", f"{label} … {read/1e6:.1f} MB (size unknown)")
                        last_beat = now
        logline("ok", f"{label} — downloaded ({read/1e6:.1f} MB).")

def on_connect(_):
    btn.disabled = True
    status_lbl.value = "connecting…"

    # 1. GPU
    section("1 / 4 · GPU")
    try:
        import torch
        if not torch.cuda.is_available():
            logline("err", "No CUDA GPU detected. Enable GPU: Runtime → Change runtime type → GPU.")
            status_lbl.value = "no GPU"
            btn.disabled = False
            return
        name = torch.cuda.get_device_name(0)
        logline("ok", f"GPU detected: {name}")
    except Exception as e:
        logline("err", f"PyTorch not importable yet: {e}")

    # 2. System deps (ffmpeg)
    section("2 / 4 · System dependencies")
    if shutil.which("ffmpeg"):
        logline("ok", "ffmpeg already present.")
    else:
        logline("run", "Installing ffmpeg…")
        rc, tail = _run("apt-get -qq update && apt-get -qq install -y ffmpeg")
        logline("ok" if rc==0 else "err",
                "ffmpeg installed." if rc==0 else f"ffmpeg install failed: {tail}")

    # 3. Python deps
    section("3 / 4 · Python packages")
    pkgs = ["opencv-python-headless", "numpy", "spandrel", "tqdm"]
    logline("run", "Installing " + ", ".join(pkgs) + " …")
    rc, tail = _run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    logline("ok" if rc==0 else "err",
            "Python packages ready." if rc==0 else f"pip failed: {tail}")

    # 4. Weights
    section("4 / 4 · Model weights (from GitHub Releases)")
    try:
        tag = _resolve_release_tag()
        logline("ok", f"Resolved release tag: {tag}")
    except Exception as e:
        logline("err", f"Could not reach GitHub API: {e}")
        status_lbl.value = "error"
        btn.disabled = False; return

    ok_all = True
    for tier, fname in WEIGHTS.items():
        dest = MS["weights_dir"] / fname
        if dest.exists() and dest.stat().st_size > 0:
            logline("ok", f"{tier} — {fname} already cached.")
            continue
        url = f"https://github.com/{GH_REPO}/releases/download/{tag}/{fname}"
        logline("run", f"{tier} — downloading {fname}…")
        try:
            _download(url, dest, tier)
        except Exception as e:
            logline("err", f"{tier} — download failed: {e}")
            ok_all = False

    if ok_all:
        MS["connected"] = True
        MS["release_tag"] = tag
        status_lbl.value = "connected"
        logline("ok", "Ready. Continue to Step 2.")
    else:
        status_lbl.value = "error"

    btn.disabled = False

btn.on_click(on_connect)

# ---------- Fix #1: display() the full layout NOW, synchronously ----------
# Running this cell renders the button live and immediately clickable — no
# intermediate step is required to "wake it up".
display(widgets.VBox([
    widgets.HBox([btn, status_lbl]),
    log_out,
]))
logline("info", "Connect cell is live. Click Connect when ready.")


## Step 2 — Upload your video

Pick a video file from your device. It's copied into the Colab VM only — nothing is sent to a third-party service.

In [ ]:
#@title 📤 Step 2 — Upload video { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
import shutil, os

MS = globals().setdefault("MOTIONSALT", {})

# Plain-text log (Fix #2), primary UI output for this cell.
_log2 = widgets.Output(layout=widgets.Layout(
    border="1px solid #e2e8f0", padding="8px",
    max_height="260px", overflow="auto",
))
_ICON2 = {"ok":"✅","warn":"⚠️","err":"❌","run":"⏳","info":"•"}
def _log2line(kind, msg):
    with _log2:
        print(f"{_ICON2.get(kind,'•')}  {msg}", flush=True)

# Fix #1: build widgets and display() synchronously on cell run.
# The picker button is live the instant this cell finishes running; the
# "connected?" check is performed AT CLICK TIME so the button is never
# stuck in a placeholder / inert state waiting on Step 1 to be re-run.
pick_btn = widgets.Button(description="Choose file…", icon="upload",
                          button_style="primary",
                          layout=widgets.Layout(width="180px", height="42px"))

def on_pick(_):
    if not MS.get("connected"):
        _log2line("err", "Run Step 1 (Connect) first — no session yet.")
        return
    from google.colab import files
    pick_btn.disabled = True
    _log2line("run", "Waiting for browser file picker…")
    try:
        uploaded = files.upload()
    except Exception as e:
        _log2line("err", f"Upload cancelled or failed: {e}")
        pick_btn.disabled = False
        return
    if not uploaded:
        _log2line("warn", "No file selected.")
        pick_btn.disabled = False
        return
    name, data = next(iter(uploaded.items()))
    src_tmp = Path("/content") / name
    dest = MS["workdir"] / "input" / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    total = len(data); read = 0
    with open(src_tmp, "rb") as fin, open(dest, "wb") as fout:
        last_pct = -1
        while True:
            chunk = fin.read(1 << 20)
            if not chunk: break
            fout.write(chunk); read += len(chunk)
            pct = int(read * 100 / max(total, 1))
            if pct >= last_pct + 10:
                _log2line("run", f"Copying into VM … {pct}%")
                last_pct = pct
    try: os.remove(src_tmp)
    except OSError: pass
    MS["input_path"] = dest
    size_mb = dest.stat().st_size / 1e6
    _log2line("ok", f"Uploaded {name} ({size_mb:.1f} MB). Continue to Step 3.")
    pick_btn.disabled = False

pick_btn.on_click(on_pick)
display(widgets.VBox([pick_btn, _log2]))
_log2line("info", "Upload cell is live. Click 'Choose file…' when ready.")
if not MS.get("connected"):
    _log2line("warn", "Step 1 (Connect) hasn't finished yet — the button will "
                     "check again at click time.")


## Step 3 — Configure & process

Pick a quality tier and dial in the filters. Then click **Start Processing**. A progress bar shows real frame-by-frame progress.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
import subprocess, shutil, math, json, time, os, sys, threading, collections, queue

MS = globals().setdefault("MOTIONSALT", {})

# =============================================================================
# Fix #1 — Widget lifecycle
# =============================================================================
# Every widget is constructed and display()-ed synchronously below.  The
# "have Steps 1 and 2 finished?" check is performed AT CLICK TIME inside
# start_processing(), never at cell-run time.  Result: running this cell
# always renders the dropdown, sliders, checkbox and Start button live and
# fully interactive; the user can adjust anything before clicking Start.
# =============================================================================

tier = widgets.Dropdown(
    options=[
        ("LOW  — AnimeJaNai V3 SuperUltraCompact (fastest)", "LOW"),
        ("MEDIUM — AnimeJaNai V3 UltraCompact (balanced)", "MEDIUM"),
        ("HIGH — Real-ESRGAN AnimeVideo v3 (best quality)", "HIGH"),
    ],
    value="MEDIUM",
    description="Quality",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="640px"),
)

def _slider(desc, default=0):
    return widgets.IntSlider(
        value=default, min=0, max=100, step=1, description=desc,
        style={"description_width": "160px"},
        layout=widgets.Layout(width="640px"),
        continuous_update=False,
    )

s_revert    = _slider("Revert Compression", 0)
s_detail    = _slider("Improve Detail", 0)
s_sharpen   = _slider("Sharpen", 15)
s_denoise   = _slider("Reduce Noise", 0)
s_dehalo    = _slider("Dehalo", 0)
s_deblur    = _slider("Anti-alias/Deblur", 0)
s_recover   = _slider("Recover Original Detail", 0)
cb_1080     = widgets.Checkbox(value=False,
                               description="Downscale to 1080p height (preserve aspect ratio)",
                               indent=False)

start = widgets.Button(description="Start Processing", icon="play",
                       button_style="success",
                       layout=widgets.Layout(width="220px", height="44px"))
frame_prog = widgets.IntProgress(value=0, min=0, max=100, description="Frames",
                                 layout=widgets.Layout(width="100%"), bar_style="info")
frame_pct  = widgets.Label(value="0/?")
stage_lbl  = widgets.Label(value="idle")

# Fix #2: plain-text log Output area, primary UI output.
log_out = widgets.Output(layout=widgets.Layout(
    border="1px solid #e2e8f0", padding="8px",
    max_height="420px", overflow="auto",
))
_ICON = {"ok":"✅","warn":"⚠️","err":"❌","run":"⏳","info":"•","perf":"📊","link":"🔗"}
def logline(kind, msg):
    icon = _ICON.get(kind, "•")
    line = f"{icon}  {msg}"
    with log_out:
        print(line, flush=True)
    if kind in ("perf", "warn", "err", "ok", "link"):
        print(f"[motionsalt/{kind}] {msg}", flush=True)

def start_processing(_):
    start.disabled = True
    # AT-CLICK-TIME preconditions (Fix #1): widgets stay live regardless of
    # step order; we validate here so the user gets a clear text message
    # instead of an inert cell.
    if not MS.get("connected"):
        logline("err", "Run Step 1 (Connect) first — no session.")
        start.disabled = False; return
    if not MS.get("input_path"):
        logline("err", "Run Step 2 (Upload) first — no input video.")
        start.disabled = False; return
    try:
        _do_process()
    except Exception as e:
        logline("err", f"Processing failed: {e}")
        raise
    finally:
        start.disabled = False

# ---------- ffprobe / nvidia-smi helpers ----------
def _ffprobe(path, args):
    out = subprocess.check_output(["ffprobe","-v","error", *args, str(path)]).decode().strip()
    return out

def _nvidia_smi_snapshot():
    try:
        out = subprocess.check_output(
            ["nvidia-smi",
             "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            stderr=subprocess.DEVNULL, timeout=1.5,
        ).decode().strip().splitlines()[0]
        util, used, total = [x.strip() for x in out.split(",")]
        return f"gpu={util}% mem={used}/{total}MiB"
    except Exception:
        return ""

def _load_model(tier_val):
    """Load the checkpoint via spandrel. fp32; scale from descriptor."""
    import torch
    from spandrel import ModelLoader, ImageModelDescriptor
    weight_file = MS["weights_map"][tier_val]
    path = MS["weights_dir"] / weight_file
    model = ModelLoader().load_from_file(str(path))
    if not isinstance(model, ImageModelDescriptor):
        raise RuntimeError(
            f"{weight_file} did not load as an ImageModelDescriptor "
            f"(got {type(model).__name__}) — cannot use this checkpoint.")
    model.cuda().eval()
    scale = int(getattr(model, "scale", 0)) or 0
    if scale <= 0:
        raise RuntimeError(
            f"Loaded {weight_file} but could not determine its upscale "
            f"factor from the descriptor (.scale={scale!r}).")
    return model, scale

# =============================================================================
# GPU-RESIDENT PIPELINE (unchanged from previous pass)
# =============================================================================
def _bgr_u8_to_rgb_f32(bgr_u8_gpu):
    import torch
    return (
        bgr_u8_gpu.flip(-1).permute(2, 0, 1).unsqueeze(0)
        .contiguous().to(torch.float32).div(255.0))

def _rgb_f32_to_bgr_u8(x):
    import torch
    return (
        x.clamp(0.0, 1.0).mul(255.0).round().to(torch.uint8)
        .squeeze(0).permute(1, 2, 0).flip(-1).contiguous())

def _gaussian_blur_gpu(x, sigma, radius=None):
    import torch, torch.nn.functional as F
    if sigma <= 0: return x
    if radius is None:
        ksize = int(round(sigma * 4.0 * 2.0 + 1.0)) | 1
        radius = (ksize - 1) // 2
    r = torch.arange(-radius, radius + 1, device=x.device, dtype=x.dtype)
    k = torch.exp(r * r / (-2.0 * sigma * sigma)); k = k / k.sum()
    kh = k.view(1, 1, 1, -1).expand(3, 1, 1, -1)
    kv = k.view(1, 1, -1, 1).expand(3, 1, -1, 1)
    xp = F.pad(x, (radius, radius, 0, 0), mode="reflect")
    x = F.conv2d(xp, kh, groups=3)
    xp = F.pad(x, (0, 0, radius, radius), mode="reflect")
    return F.conv2d(xp, kv, groups=3)

def _box_blur_3x3(x):
    import torch, torch.nn.functional as F
    xp = F.pad(x, (1, 1, 1, 1), mode="replicate")
    integ = xp.cumsum(2).cumsum(3)
    integ = F.pad(integ, (1, 0, 1, 0))
    s = (integ[:, :, 3:, 3:] - integ[:, :, :-3, 3:]
         - integ[:, :, 3:, :-3] + integ[:, :, :-3, :-3])
    return s / 9.0

def _pre_filters_gpu(t, params, _cache={}):
    import torch, torch.nn.functional as F
    out = t
    if params["revert"] > 0:
        k = params["revert"] / 100.0
        d = int(3 + 6 * k)
        lo = d // 2; hi = d - lo - 1
        sigma_color = (20.0 + 60.0 * k) / 255.0
        sigma_space = 20.0 + 40.0 * k
        cache_key = ("bilat", d, round(sigma_space, 4))
        if cache_key not in _cache:
            r = torch.arange(-lo, hi + 1, device=t.device, dtype=t.dtype)
            dy, dx = torch.meshgrid(r, r, indexing="ij")
            gs = torch.exp(-(dx * dx + dy * dy) / (2.0 * sigma_space * sigma_space))
            _cache[cache_key] = gs
        gs = _cache[cache_key]
        xp = F.pad(out, (lo, hi, lo, hi), mode="reflect")
        nb = xp.unfold(2, d, 1).unfold(3, d, 1)
        diff = nb - out.unsqueeze(-1).unsqueeze(-1)
        gc = torch.exp(-(diff * diff).sum(1) / (2.0 * sigma_color * sigma_color))
        w = gc * gs
        out = (nb * w.unsqueeze(1)).sum((-1, -2)) / w.sum((-1, -2)).unsqueeze(1).clamp_min(1e-8)
    if params["denoise"] > 0:
        h_par = 3.0 + 12.0 * (params["denoise"] / 100.0)
        sigma_c = max(h_par, 1.0) * 1.5 / 255.0
        radius = 3
        cache_key = ("nlm", radius)
        if cache_key not in _cache:
            r = torch.arange(-radius, radius + 1, device=t.device, dtype=t.dtype)
            dy, dx = torch.meshgrid(r, r, indexing="ij")
            _cache[cache_key] = torch.exp(-(dx * dx + dy * dy) / 8.0)
        gs = _cache[cache_key]
        ksz = 2 * radius + 1
        xp = F.pad(out, (radius,) * 4, mode="reflect")
        nb = xp.unfold(2, ksz, 1).unfold(3, ksz, 1)
        center_b = _box_blur_3x3(out)
        nb_flat = nb.permute(0, 4, 5, 1, 2, 3).reshape(ksz * ksz, 3, *out.shape[2:])
        nb_flat = _box_blur_3x3(nb_flat)
        nb_b = nb_flat.reshape(ksz, ksz, 1, 3, *out.shape[2:]).permute(2, 3, 4, 5, 0, 1)
        diff = nb_b - center_b.unsqueeze(-1).unsqueeze(-1)
        gc = torch.exp(-(diff * diff).sum(1) / (2.0 * sigma_c * sigma_c))
        w = gc * gs
        out = (nb * w.unsqueeze(1)).sum((-1, -2)) / w.sum((-1, -2)).unsqueeze(1).clamp_min(1e-8)
    return out

def _recover_blend_gpu(up, src, strength_0_100, scale, _cache={}):
    if strength_0_100 <= 0: return up
    import torch, torch.nn.functional as F
    _, _, H, W = up.shape
    naive = F.interpolate(src, size=(H, W), mode="bicubic",
                          align_corners=False, antialias=True)
    alpha = 0.5 * (strength_0_100 / 100.0)
    return up.lerp(naive, alpha)

def _clahe_l_gpu(L, clip_limit, tiles=(8, 8)):
    import torch, torch.nn.functional as F
    _, _, H, W = L.shape
    tx, ty = tiles
    dev, dt = L.device, L.dtype
    q = (L.clamp(0, 1) * 255.0).round().long()
    tw = (W + tx - 1) // tx
    th = (H + ty - 1) // ty
    lut = torch.empty(ty, tx, 256, device=dev, dtype=dt)
    hist_bins = torch.arange(256, device=dev)
    for gy in range(ty):
        for gx in range(tx):
            tile = q[:, :, gy * th:(gy + 1) * th, gx * tw:(gx + 1) * tw]
            n = tile.numel()
            if n == 0:
                lut[gy, gx] = hist_bins.to(dt) / 255.0
                continue
            hist = torch.bincount(tile.reshape(-1), minlength=256).to(dt)
            limit = max(clip_limit * n / 256.0, 1.0)
            excess = (hist - limit).clamp_min(0).sum()
            hist = hist.clamp(max=limit)
            hist = hist + excess / 256.0
            cdf = hist.cumsum(0)
            lut[gy, gx] = (cdf * (255.0 / max(n, 1))).round() / 255.0
    ys = torch.arange(H, device=dev, dtype=dt)
    xs = torch.arange(W, device=dev, dtype=dt)
    gyf = ys / float(th) - 0.5
    gxf = xs / float(tw) - 0.5
    gy0 = gyf.floor(); gx0 = gxf.floor()
    fy = gyf - gy0; fx = gxf - gx0
    gy1 = (gy0 + 1).clamp(max=ty - 1); gx1 = (gx0 + 1).clamp(max=tx - 1)
    gy0 = gy0.clamp(0, ty - 1); gx0 = gx0.clamp(0, tx - 1)
    qf = q.reshape(-1)
    lut_flat = lut.reshape(ty * tx, 256)
    def gather_lut(giy, gix):
        flat = (giy.long() * tx + gix.long()).reshape(-1, 1)
        return lut_flat[flat, qf.unsqueeze(1)].reshape(1, 1, H, W)
    l00 = gather_lut(gy0[:, None].expand(H, W), gx0[None, :].expand(H, W))
    l01 = gather_lut(gy0[:, None].expand(H, W), gx1[None, :].expand(H, W))
    l10 = gather_lut(gy1[:, None].expand(H, W), gx0[None, :].expand(H, W))
    l11 = gather_lut(gy1[:, None].expand(H, W), gx1[None, :].expand(H, W))
    fy4 = fy.view(1, 1, H, 1); fx4 = fx.view(1, 1, 1, W)
    top = l00 + (l01 - l00) * fx4
    bot = l10 + (l11 - l10) * fx4
    return top + (bot - top) * fy4

def _rgb_to_L_gpu(x):
    import torch
    r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
    y = 0.2126 * r + 0.7152 * g + 0.0722 * b
    eps = 216.0 / 24389.0
    kappa = 24389.0 / 27.0
    f = torch.where(y > eps, y.clamp_min(1e-12).pow(1.0 / 3.0),
                    (kappa * y + 16.0) / 116.0)
    return (116.0 * f - 16.0) / 100.0

def _post_filters_gpu(t, params):
    import torch, torch.nn.functional as F
    out = t
    if params["dehalo"] > 0:
        k = params["dehalo"] / 100.0
        gray = (0.114 * out[:, 0:1] + 0.587 * out[:, 1:2] + 0.299 * out[:, 2:3])
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]],
                          device=out.device, dtype=out.dtype).view(1, 1, 3, 3)
        ky = kx.transpose(-1, -2)
        gp = F.pad(gray, (1, 1, 1, 1), mode="reflect")
        gx = F.conv2d(gp, kx); gy = F.conv2d(gp, ky)
        mag = (gx * gx + gy * gy).sqrt()
        edges = (mag > (60.0 / 255.0 / 4.0)).to(out.dtype)
        band = -F.max_pool2d(-edges, 3, stride=1, padding=1)
        band = (band - edges).clamp_min(0.0)
        med = _box_blur_3x3(out)
        mask = band * (0.35 + 0.55 * k)
        out = out * (1.0 - mask) + med * mask
    if params["deblur"] > 0:
        k = params["deblur"] / 100.0
        sig = 0.4 + 0.9 * k
        blur = _gaussian_blur_gpu(out, sig)
        out = out * (1.0 + 0.35 * k) + blur * (-0.35 * k)
    if params["detail"] > 0:
        k = params["detail"] / 100.0
        L = _rgb_to_L_gpu(out)
        L = _clahe_l_gpu(L, clip_limit=1.0 + 2.5 * k)
        L0 = _rgb_to_L_gpu(out).clamp_min(1e-4)
        out = (out * (L / L0)).clamp(0.0, 1.0)
    if params["sharpen"] > 0:
        k = params["sharpen"] / 100.0
        blur = _gaussian_blur_gpu(out, 1.2)
        amt = 0.2 + 1.2 * k
        out = out * (1.0 + amt) + blur * (-amt)
    return out

def _infer_kernel_only(model, t, prof):
    import torch
    ev_start = torch.cuda.Event(enable_timing=True)
    ev_end   = torch.cuda.Event(enable_timing=True)
    ev_start.record()
    with torch.no_grad():
        y = model(t)
    ev_end.record()
    torch.cuda.synchronize()
    prof["infer_kernel"] += ev_start.elapsed_time(ev_end) / 1000.0
    return y

def _drain_stderr(pipe, buf):
    try:
        for line in iter(pipe.readline, b""):
            try: buf.append(line.decode("utf-8", errors="replace").rstrip())
            except Exception: buf.append(repr(line))
    except Exception: pass
    finally:
        try: pipe.close()
        except Exception: pass

def _gpu_watcher(stop_evt, samples):
    while not stop_evt.is_set():
        s = _nvidia_smi_snapshot()
        if s: samples.append((time.time(), s))
        if stop_evt.wait(3.0): return

# =============================================================================
# Fix #4 — ffmpeg -progress heartbeat helper
# =============================================================================
# Used for the audio-mux pass and the optional 1080p downscale pass.  Runs
# ffmpeg with `-progress pipe:1` (a machine-readable key=value stream) and
# a background thread parses it, emitting a plain-text heartbeat every
# ~15 s so the previously-silent ~3-minute gap becomes visible.  If parsing
# fails or the platform's ffmpeg refuses -progress, we still emit a pure
# heartbeat every 15 s.  Falls back cleanly on any error.
def _run_ffmpeg_with_progress(cmd, label, total_ms=None, beat_s=15.0):
    """cmd: list without -progress; total_ms: known duration in ms for %.
    Returns (returncode, stderr_tail)."""
    real_cmd = cmd + ["-progress", "pipe:1", "-nostats"]
    logline("run", f"{label} — starting ffmpeg…")
    proc = subprocess.Popen(
        real_cmd,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        bufsize=1, text=True,
    )
    state = {"frame": None, "out_time_ms": None, "fps": None,
             "speed": None, "last_beat": time.time(), "done": False}
    stderr_buf = collections.deque(maxlen=200)
    def _read_stderr():
        try:
            for line in iter(proc.stderr.readline, ""):
                if not line: break
                stderr_buf.append(line.rstrip())
        except Exception: pass
    st_thread = threading.Thread(target=_read_stderr, daemon=True)
    st_thread.start()

    def _read_progress():
        try:
            for line in iter(proc.stdout.readline, ""):
                if not line: break
                line = line.strip()
                if "=" not in line: continue
                k, v = line.split("=", 1)
                if k == "frame":
                    try: state["frame"] = int(v)
                    except ValueError: pass
                elif k == "out_time_ms":
                    try: state["out_time_ms"] = int(v)
                    except ValueError: pass
                elif k == "fps":
                    try: state["fps"] = float(v)
                    except ValueError: pass
                elif k == "speed":
                    state["speed"] = v.strip()
                elif k == "progress":
                    if v.strip() == "end":
                        state["done"] = True
        except Exception: pass
    pr_thread = threading.Thread(target=_read_progress, daemon=True)
    pr_thread.start()

    # Heartbeat loop.
    start_t = time.time()
    while proc.poll() is None:
        time.sleep(1.0)
        now = time.time()
        if (now - state["last_beat"]) >= beat_s:
            bits = [f"{label} still working"]
            if state["out_time_ms"] is not None:
                cur_s = state["out_time_ms"] / 1_000_000.0  # ffmpeg reports µs
                if total_ms and total_ms > 0:
                    pct = min(100, int(cur_s * 1000 / total_ms * 100))
                    bits.append(f"{pct}% ({cur_s:.1f}s / {total_ms/1000:.1f}s)")
                else:
                    bits.append(f"t={cur_s:.1f}s")
            if state["frame"] is not None:
                bits.append(f"frame={state['frame']}")
            if state["speed"]:
                bits.append(f"speed={state['speed']}")
            bits.append(f"elapsed={int(now - start_t)}s")
            logline("run", " · ".join(bits))
            state["last_beat"] = now
    pr_thread.join(timeout=1.0)
    st_thread.join(timeout=1.0)
    rc = proc.returncode
    return rc, "\n".join(stderr_buf)

# =============================================================================
# Fix #3 — shareable one-hour download link
# =============================================================================
# Choice: 0x0.st (anonymous pomf-style file host).  Justification:
#   * No account, no API key — plain HTTP POST with the file field.
#   * Free.
#   * Retention scales down with file size: files above ~0.5 GB get a
#     retention window of roughly 1 hour (the host's published policy is
#     "retention decays from 100 days for tiny files down to ~1 hour for
#     >0.5 GB uploads"), which matches the "roughly an hour, not permanent"
#     brief for a typical upscaler output.
#   * Cannot fail in a way that breaks the notebook: any exception is
#     caught and logged as a warning — the direct files.download() button
#     in Step 4 stays functional regardless.
# Fallback: transfer.sh (same interface, HTTP PUT).  If both fail we log
# a warning and move on.
def _upload_shareable(path: Path, timeout=180):
    import urllib.request, urllib.error
    size_mb = path.stat().st_size / 1e6
    logline("run", f"Creating shareable link (uploading ~{size_mb:.1f} MB, ~1 h validity)…")
    # --- primary: 0x0.st (multipart POST, no API key) ---
    try:
        import mimetypes, uuid
        boundary = "----MOTIONSALT" + uuid.uuid4().hex
        with path.open("rb") as f:
            body = f.read()
        header = (
            f"--{boundary}\r\n"
            f'Content-Disposition: form-data; name="file"; filename="{path.name}"\r\n'
            f"Content-Type: application/octet-stream\r\n\r\n"
        ).encode()
        footer = f"\r\n--{boundary}--\r\n".encode()
        data = header + body + footer
        req = urllib.request.Request(
            "https://0x0.st",
            data=data,
            headers={
                "Content-Type": f"multipart/form-data; boundary={boundary}",
                "User-Agent": "motionsalt-upscaler (colab)",
            },
        )
        with urllib.request.urlopen(req, timeout=timeout) as r:
            url = r.read().decode("utf-8").strip()
        if url.startswith("http"):
            return url, "0x0.st"
    except Exception as e:
        logline("warn", f"0x0.st upload failed ({e}); trying transfer.sh…")
    # --- fallback: transfer.sh (HTTP PUT) ---
    try:
        with path.open("rb") as f:
            body = f.read()
        req = urllib.request.Request(
            f"https://transfer.sh/{path.name}",
            data=body, method="PUT",
            headers={"User-Agent": "motionsalt-upscaler (colab)"},
        )
        with urllib.request.urlopen(req, timeout=timeout) as r:
            url = r.read().decode("utf-8").strip()
        if url.startswith("http"):
            return url, "transfer.sh"
    except Exception as e:
        logline("warn", f"transfer.sh upload failed ({e}).")
    return None, None

# =============================================================================
# Main pipeline
# =============================================================================
def _do_process():
    import cv2, numpy as np, torch
    in_path = MS["input_path"]
    out_dir = MS["workdir"] / "out"; out_dir.mkdir(exist_ok=True)
    stem = in_path.stem

    # 1. Probe input
    stage_lbl.value = "Reading source metadata…"
    fps = _ffprobe(in_path, ["-select_streams","v:0","-show_entries","stream=r_frame_rate","-of","csv=p=0"])
    num, den = (fps.split("/") + ["1"])[:2]
    fps_f = float(num) / float(den) if float(den) else 30.0
    nframes = int(_ffprobe(in_path, ["-select_streams","v:0","-count_packets","-show_entries","stream=nb_read_packets","-of","csv=p=0"]) or "0")
    # source duration (ms) for the ffmpeg -progress percentage in mux/downscale
    try:
        dur_s = float(_ffprobe(in_path, ["-show_entries","format=duration","-of","csv=p=0"]) or "0")
    except Exception:
        dur_s = 0.0
    total_ms = int(dur_s * 1000) if dur_s > 0 else None
    logline("ok", f"Source: {fps_f:.3f} fps · ~{nframes or 'unknown'} frames · duration ~{dur_s:.1f}s.")

    # 2. Load model
    stage_lbl.value = "Loading model…"
    logline("run", f"Loading {tier.value} model…")
    cuda_ok = torch.cuda.is_available()
    logline("info", f"torch.cuda.is_available() = {cuda_ok} · torch={torch.__version__} · "
                    f"device_count={torch.cuda.device_count() if cuda_ok else 0}")
    if not cuda_ok:
        raise RuntimeError("CUDA is not available — refusing to run on CPU. "
                           "Reconnect to a GPU runtime and re-run Step 1.")
    torch.backends.cudnn.benchmark = True
    model, scale = _load_model(tier.value)
    gpu_name = torch.cuda.get_device_name(0)
    logline("ok", f"{tier.value} model loaded on {gpu_name} · native scale {scale}× · fp32.")
    try:
        first_param = next(model.model.parameters()) if hasattr(model, "model") else next(model.parameters())
    except Exception:
        first_param = None
    if first_param is None:
        raise RuntimeError("Could not read model parameters to confirm device placement.")
    logline("info", f"First model param device = {first_param.device} · dtype = {first_param.dtype}")
    if first_param.device.type != "cuda":
        raise RuntimeError(
            f"Model parameters ended up on {first_param.device} instead of "
            f"cuda after .cuda() — refusing to run.")

    pre_snap = _nvidia_smi_snapshot()
    if pre_snap: logline("info", f"nvidia-smi (pre-run): {pre_snap}")

    params = dict(
        revert=s_revert.value, detail=s_detail.value, sharpen=s_sharpen.value,
        denoise=s_denoise.value, dehalo=s_dehalo.value, deblur=s_deblur.value,
        recover=s_recover.value,
    )
    logline("info",
            "Pipeline: frame → GPU (one H2D), pre/infer/recover/post all on GPU, "
            "one D2H of packed BGR uint8 into ffmpeg. No numpy/cv2 per frame.")

    # 3. Open source + ffmpeg pipe
    cap = cv2.VideoCapture(str(in_path))
    if not cap.isOpened():
        raise RuntimeError("OpenCV could not open the input video.")
    w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if w <= 0 or h <= 0:
        raise RuntimeError(f"OpenCV reported invalid source dimensions {w}x{h}.")
    out_w, out_h = w * scale, h * scale
    crop_w = w - (out_w % 2 != 0); crop_h = h - (out_h % 2 != 0)
    if (crop_w, crop_h) != (w, h):
        logline("warn",
                f"Source {w}x{h} at scale {scale}× → odd output "
                f"({out_w}x{out_h}); cropping source to {crop_w}x{crop_h}.")
        w, h = crop_w, crop_h
        out_w, out_h = w * scale, h * scale
    if out_w % 2 or out_h % 2:
        raise RuntimeError(f"Refusing to write odd output dims {out_w}x{out_h}.")

    stage_lbl.value = f"Upscaling {w}×{h} → {out_w}×{out_h} ({scale}×, fp32)"

    video_only = out_dir / f"{stem}_upscaled_noaudio.mp4"

    ffmpeg_cmd = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-f", "rawvideo", "-vcodec", "rawvideo", "-pix_fmt", "bgr24",
        "-s", f"{out_w}x{out_h}", "-r", f"{fps_f}", "-an", "-i", "-",
        "-c:v", "libx264", "-preset", "medium", "-crf", "16",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(video_only),
    ]
    logline("info", "ffmpeg: " + " ".join(ffmpeg_cmd))

    ff = subprocess.Popen(
        ffmpeg_cmd, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE, bufsize=0,
    )
    stderr_buf = collections.deque(maxlen=200)
    stderr_thread = threading.Thread(target=_drain_stderr, args=(ff.stderr, stderr_buf), daemon=True)
    stderr_thread.start()
    time.sleep(0.25)
    if ff.poll() is not None:
        stderr_thread.join(timeout=1.0)
        tail = "\n".join(stderr_buf) or "(no stderr captured)"
        raise RuntimeError(f"ffmpeg exited immediately (returncode={ff.returncode}). stderr:\n{tail}")

    expected_frame_bytes = out_w * out_h * 3
    frame_prog.max = max(nframes, 1)
    i = 0; t0 = time.time()

    stage_times = collections.defaultdict(float)
    report_at = {3, 5, 10, 15, 20, 30, 50, 100}
    buckets_ordered = ("read", "pre", "infer_h2d", "infer_kernel",
                       "infer_d2h", "recover", "post", "write")

    gpu_samples = collections.deque(maxlen=200)
    gpu_stop = threading.Event()
    gpu_thread = threading.Thread(target=_gpu_watcher, args=(gpu_stop, gpu_samples), daemon=True)
    gpu_thread.start()

    last_ui_update = 0.0; last_stdout_beat = 0.0; last_gpu_log = 0.0

    need_pre  = (params["revert"] or params["denoise"])
    need_post = (params["dehalo"] or params["deblur"] or params["detail"] or params["sharpen"])
    need_rec  = params["recover"] > 0
    gpu_stage = None

    try:
        while True:
            s = time.perf_counter()
            ok, frame = cap.read()
            if not ok: break
            if frame.shape[1] != w or frame.shape[0] != h:
                frame = frame[:h, :w]
            stage_times["read"] += time.perf_counter() - s

            s = time.perf_counter()
            if gpu_stage is None or gpu_stage.shape[0] != frame.shape[0] or gpu_stage.shape[1] != frame.shape[1]:
                gpu_stage = torch.empty(frame.shape, dtype=torch.uint8, device="cuda")
            gpu_stage.copy_(torch.from_numpy(frame), non_blocking=False)
            t_in = _bgr_u8_to_rgb_f32(gpu_stage)
            torch.cuda.synchronize()
            stage_times["infer_h2d"] += time.perf_counter() - s

            s = time.perf_counter()
            if need_pre:
                t_in = _pre_filters_gpu(t_in, params)
                torch.cuda.synchronize()
            stage_times["pre"] += time.perf_counter() - s

            y = _infer_kernel_only(model, t_in, stage_times)

            s = time.perf_counter()
            if need_rec:
                y = _recover_blend_gpu(y, t_in, params["recover"], scale)
                torch.cuda.synchronize()
            stage_times["recover"] += time.perf_counter() - s

            s = time.perf_counter()
            if need_post:
                y = _post_filters_gpu(y, params)
                torch.cuda.synchronize()
            stage_times["post"] += time.perf_counter() - s

            s = time.perf_counter()
            out_u8 = _rgb_f32_to_bgr_u8(y)
            buf = out_u8.cpu().numpy().tobytes()
            stage_times["infer_d2h"] += time.perf_counter() - s

            if len(buf) != expected_frame_bytes:
                raise RuntimeError(
                    f"Frame {i}: byte length {len(buf)} != expected "
                    f"{expected_frame_bytes} (source {h}x{w}, model native scale {scale}×).")

            s = time.perf_counter()
            try:
                ff.stdin.write(buf)
            except BrokenPipeError:
                ff.wait(timeout=2.0); stderr_thread.join(timeout=1.0)
                tail = "\n".join(stderr_buf) or "(no stderr captured)"
                raise RuntimeError(f"Broken pipe while writing frame {i} to ffmpeg "
                                   f"(returncode={ff.returncode}). stderr:\n{tail}")
            stage_times["write"] += time.perf_counter() - s
            i += 1

            if i in report_at:
                total = sum(stage_times[k] for k in buckets_ordered) or 1e-9
                parts = [f"{k}={stage_times[k]/i*1000:.1f}ms ({stage_times[k]/total*100:.0f}%)"
                         for k in buckets_ordered]
                fps_now = i / max(time.time() - t0, 1e-6)
                logline("perf", f"[frame {i}] {fps_now:.3f} fps · " + " · ".join(parts))

            now = time.time()
            if (now - last_gpu_log) >= 6.0 and gpu_samples:
                _, snap = gpu_samples[-1]
                logline("perf", f"nvidia-smi: {snap} @ frame {i}")
                last_gpu_log = now
            if i == 1 or (now - last_ui_update) >= 0.5 or i == nframes:
                frame_prog.value = min(i, frame_prog.max)
                elapsed = now - t0
                fps_now = i / max(elapsed, 1e-6)
                eta = (nframes - i) / fps_now if (nframes and fps_now > 0) else 0
                eta_str = f" · ETA {int(eta//60)}m{int(eta%60):02d}s" if eta else ""
                frame_pct.value = f"{i}/{nframes or '?'} · {fps_now:.2f} fps{eta_str}"
                last_ui_update = now
            if (now - last_stdout_beat) >= 10.0:
                print(f"[motionsalt] frame {i}/{nframes or '?'}"
                      f"({i/max(now-t0,1e-6):.2f} fps)", flush=True)
                last_stdout_beat = now
    finally:
        cap.release()
        try:
            if ff.stdin and not ff.stdin.closed: ff.stdin.close()
        except Exception: pass
        gpu_stop.set()

    rc = ff.wait()
    stderr_thread.join(timeout=2.0)
    gpu_thread.join(timeout=4.0)
    if rc != 0:
        tail = "\n".join(stderr_buf) or "(no stderr captured)"
        raise RuntimeError(f"ffmpeg exited with code {rc} after writing {i} frames. stderr:\n{tail}")

    total = sum(stage_times[k] for k in buckets_ordered) or 1e-9
    wallclock = time.time() - t0
    final_fps = i / max(wallclock, 1e-6)
    logline("ok", f"Upscaled {i} frames in {wallclock:.1f}s ({final_fps:.3f} fps).")
    logline("perf", f"Per-stage average (ms/frame) over {i} frames, {final_fps:.3f} fps wall:")
    for k in buckets_ordered:
        v = stage_times[k]
        logline("perf", f"  {k:14s}: {v/i*1000:8.1f} ms/frame  ({v/total*100:4.1f}%)")
    if gpu_samples:
        utils = []
        for _, s in gpu_samples:
            try: utils.append(int(s.split("gpu=")[1].split("%")[0]))
            except Exception: pass
        if utils:
            logline("perf",
                    f"  GPU util range: min={min(utils)}% max={max(utils)}% "
                    f"avg={sum(utils)//len(utils)}% across {len(utils)} samples")

    # 4. Audio mux — with -progress heartbeat (Fix #4)
    stage_lbl.value = "Muxing original audio…"
    logline("run", "Starting audio mux (typically up to ~3 minutes).")
    with_audio = out_dir / f"{stem}_upscaled.mp4"
    mux_cmd = [
        "ffmpeg","-y","-hide_banner","-loglevel","error",
        "-i",str(video_only),"-i",str(in_path),
        "-map","0:v:0","-map","1:a:0?","-c:v","copy","-c:a","aac","-b:a","192k",
        "-shortest", str(with_audio),
    ]
    rc, stderr_tail = _run_ffmpeg_with_progress(mux_cmd, "Audio mux",
                                                total_ms=total_ms, beat_s=15.0)
    if rc != 0:
        if stderr_tail:
            logline("warn", f"Audio mux failed: {stderr_tail.strip().splitlines()[-1]}")
        try:
            shutil.move(str(video_only), str(with_audio))
            logline("warn","Source had no audio track (or codec mismatch); output is silent.")
        except Exception as e:
            logline("err", f"Could not finalize output: {e}")
            raise
    else:
        try: os.remove(video_only)
        except OSError: pass
        logline("ok","Audio muxed back in.")

    final = with_audio

    # 5. Optional 1080p downscale — same heartbeat treatment
    if cb_1080.value:
        stage_lbl.value = "Downscaling to 1080p height (aspect-preserving)…"
        logline("run", "Starting 1080p downscale.")
        down = out_dir / f"{stem}_upscaled_1080p.mp4"
        down_cmd = [
            "ffmpeg","-y","-hide_banner","-loglevel","error",
            "-i",str(final),
            "-vf","scale=-2:1080:flags=lanczos",
            "-c:v","libx264","-preset","medium","-crf","17","-pix_fmt","yuv420p",
            "-c:a","copy", str(down),
        ]
        rc, stderr_tail = _run_ffmpeg_with_progress(down_cmd, "1080p downscale",
                                                    total_ms=total_ms, beat_s=15.0)
        if rc == 0:
            final = down
            logline("ok","Downscaled to 1080p height, aspect ratio preserved.")
        else:
            last = stderr_tail.strip().splitlines()[-1] if stderr_tail.strip() else "unknown"
            logline("warn", f"1080p downscale failed ({last}) — keeping full-res output.")

    MS["output_path"] = final
    stage_lbl.value = f"Done. Output: {final.name}"
    logline("ok", f"Ready for Step 4. File: {final.name} · {final.stat().st_size/1e6:.1f} MB.")

    # 6. Shareable link (Fix #3) — additive; direct download button in Step 4
    #    keeps working regardless of whether this succeeds.
    MS.pop("shareable_url", None)
    MS.pop("shareable_host", None)
    try:
        url, host = _upload_shareable(final)
        if url:
            MS["shareable_url"] = url
            MS["shareable_host"] = host
            logline("link", f"Shareable link (valid ~1 hour, via {host}):")
            logline("link", url)
            logline("info", "Copy the link above — you can open it on any device/browser "
                            "to download the file. It will stop working after roughly one hour.")
        else:
            logline("warn", "Could not create a shareable link. The direct "
                            "download button in Step 4 still works normally.")
    except Exception as e:
        logline("warn", f"Shareable-link step errored: {e}. Direct download in Step 4 unaffected.")

start.on_click(start_processing)

# Fix #1: display the whole form synchronously, right now, so widgets are
# fully live the moment this cell finishes running.  Any missing precondition
# (Step 1 not done, Step 2 not done) is surfaced only when Start is clicked.
display(widgets.VBox([
    tier,
    s_revert, s_detail, s_sharpen, s_denoise, s_dehalo, s_deblur, s_recover,
    cb_1080,
    start,
    stage_lbl,
    widgets.HBox([frame_prog, frame_pct]),
    log_out,
]))
logline("info", "Configure & Process cell is live. Adjust freely; work starts "
                "only when you click Start Processing.")
if not MS.get("connected"):
    logline("warn", "Step 1 (Connect) hasn't been run yet — the Start button "
                    "will check again at click time.")
if not MS.get("input_path"):
    logline("warn", "Step 2 (Upload) hasn't been run yet — the Start button "
                    "will check again at click time.")


## Step 4 — Download

Click the button to download the file directly through your browser. If Step 3 succeeded in creating a **shareable link** (~1 hour validity), it will also appear in the log below — copy it to open on any other device or browser.


In [ ]:
#@title ⬇️ Step 4 — Download result { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display

MS = globals().setdefault("MOTIONSALT", {})

# Plain-text log (Fix #2), primary UI output.
_log4 = widgets.Output(layout=widgets.Layout(
    border="1px solid #e2e8f0", padding="8px",
    max_height="220px", overflow="auto",
))
_ICON4 = {"ok":"✅","warn":"⚠️","err":"❌","run":"⏳","info":"•","link":"🔗"}
def _log4line(kind, msg):
    with _log4:
        print(f"{_ICON4.get(kind,'•')}  {msg}", flush=True)

# Fix #1: the button exists and is clickable as soon as the cell renders.
# Whether there is an output file to serve is checked AT CLICK TIME, so this
# cell never sits in a placeholder/inert state waiting on Step 3 to be re-run.
dl_btn = widgets.Button(description="⬇️ Download Result", button_style="primary",
                        layout=widgets.Layout(width="240px", height="46px"))

def on_click(_):
    out = MS.get("output_path")
    if not out:
        _log4line("err", "No output yet — run Step 3 (Configure & process) first.")
        return
    from google.colab import files
    dl_btn.disabled = True
    _log4line("run", f"Preparing browser download for {out.name} "
                     f"({out.stat().st_size/1e6:.1f} MB)…")
    try:
        files.download(str(out))
        _log4line("ok", "Download started in your browser.")
    except Exception as e:
        _log4line("err", f"Browser download failed: {e}")
    finally:
        dl_btn.disabled = False

dl_btn.on_click(on_click)
display(widgets.VBox([dl_btn, _log4]))

# Surface current status + shareable link (from Step 3) as plain text.
out = MS.get("output_path")
if out:
    _log4line("ok", f"File ready: {out.name} · {out.stat().st_size/1e6:.1f} MB")
else:
    _log4line("warn", "No output on record yet. Run Step 3 first; then click "
                     "the Download Result button above.")

url = MS.get("shareable_url")
if url:
    host = MS.get("shareable_host") or "external host"
    _log4line("link", f"Shareable link (valid ~1 hour, via {host}):")
    _log4line("link", url)
    _log4line("info", "Copy that link — open it in any browser/device to "
                     "download the file. Expires after roughly one hour.")
else:
    _log4line("info", "No shareable link on record (may have failed or Step 3 "
                     "wasn't run this session). The direct download button "
                     "above always works.")


---

<div align="center" style="font-family:-apple-system,Segoe UI,sans-serif;color:#64748b;font-size:12px;padding:10px;">
MOTIONSALT Upscaler · MIT-licensed wrapper · powered by
<a href="https://github.com/the-database/mpv-upscale-2x_animejanai">AnimeJaNai V3</a> and
<a href="https://github.com/xinntao/Real-ESRGAN">Real-ESRGAN AnimeVideo v3</a>.<br>
Source: <a href="https://github.com/motionssalt/upscale">github.com/motionssalt/upscale</a>
</div>